# **Durability — Neural Network DATASET GENERATION (from the PCE, not the emulator)**

This notebook loads the `pce_metamodel` files, one per time step, and queries them at fresh design
points to build a denser dataset for training a single global NN surrogate — the durability
counterpart of [`03_generate_dataset_nn.ipynb`](../benchmark/03_generate_dataset_nn.ipynb) in the
benchmark folder.

## **1. Libraries**

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Uniform, JointIndependent

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Config**

Must match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) /
[`02_train_pce.ipynb`](02_train_pce.ipynb) — `fck_min`/`fck_max`/`rh_min`/`rh_max`/`cov_min`/`cov_max`
rebuild the same `joint`, `times` must be the same grid, and `n_latent_samples` /
`installation_year` / `co2_scenario` / `cement_type` / `exposure_conditions` name the
`pce_metamodel` files being loaded below.

`n_points` is new: how many fresh `(fck, rh, cov)` points to query per time step. Since a PCE
evaluation is cheap, this can be far denser than the design sample count used to fit the PCEs
themselves.

In [2]:
fck_min = 20  # MPa
fck_max = 50  # MPa
rh_min  = 20  # %
rh_max  = 80  # %
cov_min = 15  # mm
cov_max = 60  # mm

n_latent_samples     = 2500     # must match stage 1/2 — it names the pce_metamodel files
installation_year    = 1980     # must match stage 1/2
co2_scenario         = "SSP2-4.5"  # must match stage 1/2
cement_type          = 3        # must match stage 1/2
exposure_conditions  = 2        # must match stage 1/2
n_lambdas            = 4
n_points             = 5000     # (fck, rh, cov) query points drawn per time step for the NN dataset

lambda3_fixed = 0.198312  # written by hand — same values as 02_train_pce_plot.ipynb, the PCE's own lambda 3 / lambda 4 are unreliable
lambda4_fixed = 0.130039

times = np.linspace(0, 100, 5, endpoint=True)  # must match stage 1/2
times

array([  0.,  25.,  50.,  75., 100.])

## 3. Rebuild the joint distribution

In [3]:
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

## 4. Load the per-time-step PCE metamodels

One `pce_metamodel` per entry of `times`, as saved by `train_and_validate_pce_from_dataset_durability`
in stage 2.

In [4]:
pce_models = []
for t in times:
    tag = f'{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
    with open(f'{n_latent_samples}_pce_metamodel_{tag}.pkl', 'rb') as f:
        pce_models.append(dill.load(f))

print(f"Loaded {len(pce_models)} PCE metamodels")

Loaded 5 PCE metamodels


## 5. Query the PCEs and stack the dataset

`generate_nn_dataset_durability` draws `n_points` fresh `(fck, rh, cov)` samples per time step,
evaluates the matching PCE on them, and stacks every time step into one dataframe with an explicit
`Time (years)` column. `lambda 1`/`lambda 2` come from the PCE prediction; `lambda 3`/`lambda 4` are
overridden with the hand-written `lambda3_fixed`/`lambda4_fixed` from section 2.

In [5]:
print("="*60)
print("GENERATING THE NN DATASET FROM THE PCE MODELS")
print("="*60)

result = generate_nn_dataset_durability(
                                           pce_metamodels=pce_models,
                                           times=times,
                                           joint=joint,
                                           installation_year=installation_year,
                                           cement_type=cement_type,
                                           exposure_conditions=exposure_conditions,
                                           co2_scenario=co2_scenario,
                                           n_points=n_points,
                                           n_lambdas=n_lambdas,
                                           lambda3_fixed=lambda3_fixed,
                                           lambda4_fixed=lambda4_fixed,
                                           n_latent_samples=n_latent_samples,
                                           output_dir='.',
                                        )

df_nn = result['dataset_nn']
print(f"\nTotal rows: {len(df_nn)}")
df_nn.head()

GENERATING THE NN DATASET FROM THE PCE MODELS

----------------------------------------
GENERATING NN DATASET FROM 5 PCE MODELS
----------------------------------------
  lambda 3 fixed at 0.1983
  lambda 4 fixed at 0.1300
  t = 0.00 years: 5000 points queried from the PCE
  t = 25.00 years: 5000 points queried from the PCE
  t = 50.00 years: 5000 points queried from the PCE
  t = 75.00 years: 5000 points queried from the PCE
  t = 100.00 years: 5000 points queried from the PCE
The NN dataset has been saved!

Total rows: 25000


,fck,rh,cov,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,33.769371,74.863008,42.458042,0.0,41.897836,0.228203,0.198312,0.130039
1,35.574994,41.697355,46.897049,0.0,46.289976,0.208585,0.198312,0.130039
2,38.208432,63.329779,15.641013,0.0,15.438305,0.610554,0.198312,0.130039
3,42.201708,73.868358,39.323328,0.0,38.789182,0.243619,0.198312,0.130039
4,22.701543,63.286029,15.058318,0.0,14.824194,0.630618,0.198312,0.130039


## 6. Sanity check — GLD validity

In [6]:
n_before = len(df_nn)
invalid  = df_nn['lambda 2'] <= 0

print(f'rows before : {n_before}')
print(f'invalid     : {invalid.sum()} ({invalid.mean() * 100:.3f}%)')

if invalid.any():
    print('\nregion of the input space where the PCE produced an invalid GLD:')
    print(df_nn.loc[invalid, ['fck', 'rh', 'cov', 'Time (years)', 'lambda 1', 'lambda 2']].describe().loc[
              ['min', 'max']].round(3).to_string())
    df_nn = df_nn.loc[~invalid].reset_index(drop=True)

print(f'\nrows after  : {len(df_nn)}')
assert (df_nn['lambda 2'] > 0).all(), 'lambda 2 must be strictly positive'
df_nn.describe()

rows before : 25000
invalid     : 0 (0.000%)

rows after  : 25000


,fck,rh,cov,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,2.500000e+04,25000.000000
mean,35.021737,49.785043,37.402315,50.000000,25.400022,0.245077,1.983120e-01,0.130039
std,8.677355,17.273498,13.017464,35.356046,15.665993,0.097510,2.775613e-17,0.000000
min,20.000422,20.009421,15.004570,0.000000,-29.316878,0.065698,1.983120e-01,0.130039
25%,27.473843,34.777643,26.099891,25.000000,14.331523,0.176494,1.983120e-01,0.130039
50%,35.065331,49.797020,37.356542,50.000000,25.232855,0.215637,1.983120e-01,0.130039
75%,42.529563,64.533169,48.636606,75.000000,37.334168,0.288930,1.983120e-01,0.130039
max,49.999557,79.997805,59.999900,100.000000,59.137050,0.630618,1.983120e-01,0.130039


In [7]:
# re-save the validated dataset under the same name the training notebook reads
tag = f'install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
with open(f'{n_latent_samples}_dataset_nn_durability_{tag}.pkl', 'wb') as f:
    dill.dump(df_nn, f)

print(f'saved {len(df_nn)} validated rows to {n_latent_samples}_dataset_nn_durability_{tag}.pkl')

saved 25000 validated rows to 2500_dataset_nn_durability_install_1980_cement_3_exposure_2_co2_SSP2-4.5.pkl
